# Notebook 2 — Create the Labels

## Objective

Create a binary delivery label for each order by comparing the actual delivery date
with the estimated delivery date.

- `Late` → actual delivery was after the estimated delivery date
- `On Time` → actual delivery was on or before the estimated delivery date

We will also validate the label on real orders and examine class balance.

## Artifact

`artifacts/labeled_table.parquet`

# Imports and load the ML table

In [1]:
import pandas as pd

ml_table = pd.read_parquet("../artifacts/ml_table.parquet")

print("Shape:", ml_table.shape)
display(ml_table.head())

Shape: (99441, 19)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,item_count,product_count,seller_count,total_price,total_freight,payment_count,payment_value,max_installments
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,1.0,1.0,1.0,29.99,8.72,3.0,38.71,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,barreiras,BA,1.0,1.0,1.0,118.70,22.76,1.0,141.46,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,1.0,1.0,1.0,159.90,19.22,1.0,179.12,3.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,1.0,1.0,1.0,45.00,27.20,1.0,72.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,1.0,1.0,1.0,19.90,8.72,1.0,28.62,1.0


In [2]:
print(
    ml_table[
        ["order_id", "order_delivered_customer_date",
         "order_estimated_delivery_date"]
    ].head()
)

                           order_id order_delivered_customer_date  \
0  e481f51cbdc54678b7cc49136f2d6af7           2017-10-10 21:25:13   
1  53cdb2fc8bc7dce0b6741e2150273451           2018-08-07 15:27:45   
2  47770eb9100c2d0c44946d9cf07ec65d           2018-08-17 18:06:29   
3  949d5b44dbf5de918fe9c16f97b45f8a           2017-12-02 00:28:42   
4  ad21c59c0840e6cb83a9ceb5573f8159           2018-02-16 18:17:02   

  order_estimated_delivery_date  
0           2017-10-18 00:00:00  
1           2018-08-13 00:00:00  
2           2018-09-04 00:00:00  
3           2017-12-15 00:00:00  
4           2018-02-26 00:00:00  


# Convert dates

In [3]:
ml_table["order_delivered_customer_date"] = pd.to_datetime(
    ml_table["order_delivered_customer_date"],
    errors="coerce"
)

ml_table["order_estimated_delivery_date"] = pd.to_datetime(
    ml_table["order_estimated_delivery_date"],
    errors="coerce"
)

In [4]:
print(ml_table[
    ["order_delivered_customer_date", "order_estimated_delivery_date"]
].dtypes)

order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


# Check missing dates

In [5]:
print("Missing delivered date:",
      ml_table["order_delivered_customer_date"].isna().sum())

print("Missing estimated date:",
      ml_table["order_estimated_delivery_date"].isna().sum())

Missing delivered date: 2965
Missing estimated date: 0


In [6]:
labeled_table = ml_table.dropna(
    subset=[
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
).copy()

print("Rows after removing missing dates:", len(labeled_table))

Rows after removing missing dates: 96476


# Create the label

In [7]:
labeled_table["label"] = (
    labeled_table["order_delivered_customer_date"]
    > labeled_table["order_estimated_delivery_date"]
).map({
    True: "Late",
    False: "On Time"
})

In [8]:
display(
    labeled_table[
        [
            "order_id",
            "order_delivered_customer_date",
            "order_estimated_delivery_date",
            "label"
        ]
    ].head(10)
)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,label
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-10 21:25:13,2017-10-18,On Time
1,53cdb2fc8bc7dce0b6741e2150273451,2018-08-07 15:27:45,2018-08-13,On Time
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-17 18:06:29,2018-09-04,On Time
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-12-02 00:28:42,2017-12-15,On Time
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-16 18:17:02,2018-02-26,On Time
5,a4591c265e18cb1dcee52889e2d8acc3,2017-07-26 10:57:55,2017-08-01,On Time
7,6514b8ad8028c9f2cc2374ded245783f,2017-05-26 12:55:51,2017-06-07,On Time
8,76c6e866289321a7c93b82b54852dc33,2017-02-02 14:08:10,2017-03-06,On Time
9,e69bfb5eb88e0ed6a785585b27e16dbf,2017-08-16 17:14:30,2017-08-23,On Time
10,e6ce16cb79ec1d90b1da9085a6118aeb,2017-05-29 11:18:31,2017-06-07,On Time


# Validate the label on real orders

In [9]:
sample = labeled_table[
    [
        "order_id",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "label"
    ]
].sample(10, random_state=42)

display(sample)

,order_id,order_delivered_customer_date,order_estimated_delivery_date,label
22500,c58cff333993bb6b7161d7ec1350eef3,2018-04-06 02:32:49,2018-04-18,On Time
68942,87673b5ccb20de0a91c28cc461105d76,2018-05-23 15:28:28,2018-05-30,On Time
23988,80b430d0029bb33110ac31d60e87e0b8,2017-12-07 18:43:46,2017-12-27,On Time
31304,580603672a21252f21fa8a8b4ca85986,2018-04-26 17:44:27,2018-05-08,On Time
36131,c09f32e7ba9b4a134455b36eeff8fff3,2018-04-13 17:32:07,2018-04-24,On Time
58331,462240cb1ec4e5db517e73fff57ebfc0,2017-12-14 18:56:14,2017-12-20,On Time
95960,5bc2a7b8f0817443a86b69461e742cc9,2018-01-12 21:59:18,2018-02-01,On Time
56132,4ef8f514f95bb0a41f58965ea04e7027,2017-05-26 15:59:47,2017-06-07,On Time
56751,587904dc1c873ebfb6078160b4819d8e,2017-12-06 19:43:34,2017-12-15,On Time
92212,29c3b79aace1b72a82b1232bf494e16f,2018-04-28 15:51:50,2018-01-24,Late


In [10]:
check = (
    (labeled_table["order_delivered_customer_date"]
     > labeled_table["order_estimated_delivery_date"])
    == (labeled_table["label"] == "Late")
)

print("Incorrect labels:", (~check).sum())

assert check.all()

print("Label logic verified")

Incorrect labels: 0
Label logic verified


# Look at class distribution

In [11]:
class_counts = labeled_table["label"].value_counts()

print(class_counts)

label
On Time    88649
Late        7827
Name: count, dtype: int64


In [12]:
class_percent = (
    labeled_table["label"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(class_percent)

label
On Time    91.89
Late        8.11
Name: proportion, dtype: float64


# Decide whether there is class imbalance

In [13]:
majority = class_counts.max()
minority = class_counts.min()

imbalance_ratio = majority / minority

print("Imbalance ratio:", round(imbalance_ratio, 2))

Imbalance ratio: 11.33


In [14]:
print("Class distribution:")
for label, percentage in class_percent.items():
    print(f"{label}: {percentage}%")

Class distribution:
On Time: 91.89%
Late: 8.11%


## Class Imbalance Assessment

The dataset contains 91.89% On Time orders and 8.11% Late orders.

The imbalance ratio is 11.33, meaning there are approximately 11.33 On Time orders
for every Late order.

This represents a severe class imbalance. Because the Late class is the minority
class, accuracy alone may be misleading when evaluating the models. Later model
evaluation should therefore also consider precision, recall, F1-score, and
ROC-AUC/PR-AUC, with particular attention to the model's ability to identify Late
orders.

# Final validation

In [15]:
print("Rows:", len(labeled_table))
print("Unique orders:", labeled_table["order_id"].nunique())
print("Duplicate orders:", labeled_table["order_id"].duplicated().sum())

assert labeled_table["order_id"].is_unique
assert labeled_table["label"].notna().all()

print("Labeled table validated")

Rows: 96476
Unique orders: 96476
Duplicate orders: 0
Labeled table validated


# Save the artifact

In [16]:
labeled_table.to_parquet(
    "../artifacts/labeled_table.parquet",
    index=False
)

print("labeled_table.parquet saved")

labeled_table.parquet saved


In [18]:
saved = pd.read_parquet(
    "../artifacts/labeled_table.parquet"
)

print("Shape:", saved.shape)
print("Unique orders:", saved["order_id"].nunique())
print("Labels:", saved["label"].value_counts())

assert saved["order_id"].is_unique

print("Labeled artifact verified")

Shape: (96476, 20)
Unique orders: 96476
Labels: label
On Time    88649
Late        7827
Name: count, dtype: int64
Labeled artifact verified
